In [4]:
import xarray as xr
import pandas as pd
from pathlib import Path

# --- bbox (same as kelp) ---
lat_min, lat_max = 38.0, 40.0
lon_min, lon_max = -125.0, -123.0

# --- find files ---
oisst_dir = Path("../1_DATA/raw")  # adjust if your OISST files are in raw/oisst/
files = sorted(oisst_dir.rglob("oisst_socal_*.nc"))
if not files:
    raise FileNotFoundError(f"No files matched under {oisst_dir.resolve()}")

print("Found", len(files), "files")
print("First:", files[0].name, "| Last:", files[-1].name)

out_csv = Path("../1_DATA/processed/oisst_NORCAL_bbox_monthly.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

# --- open all files ---
ds = xr.open_mfdataset(files, combine="by_coords")

# --- detect coord/var names ---
def pick_first(existing, candidates):
    for c in candidates:
        if c in existing:
            return c
    return None

lat_name = pick_first(ds.coords, ["lat", "latitude"])
lon_name = pick_first(ds.coords, ["lon", "longitude"])
time_name = pick_first(ds.coords, ["time"])

sst_var = pick_first(ds.data_vars, ["sst", "SST", "analysed_sst", "sea_surface_temperature"])
if None in (lat_name, lon_name, time_name, sst_var):
    raise ValueError(f"Could not detect names. coords={list(ds.coords)}, data_vars={list(ds.data_vars)}")

# --- normalize lon to [-180, 180] if needed ---
if float(ds[lon_name].max()) > 180:
    ds = ds.assign_coords({lon_name: (((ds[lon_name] + 180) % 360) - 180)}).sortby(lon_name)

sst = ds[sst_var]

# --- subset bbox (handle descending lat) ---
sst_bbox = sst.sel({lat_name: slice(lat_min, lat_max), lon_name: slice(lon_min, lon_max)})
if sst_bbox.sizes.get(lat_name, 0) == 0:
    sst_bbox = sst.sel({lat_name: slice(lat_max, lat_min), lon_name: slice(lon_min, lon_max)})

# --- bbox mean (should now be 1D time) ---
sst_mean = sst_bbox.mean(dim=[lat_name, lon_name], skipna=True).squeeze()

# --- to pandas series ---
sst_daily = sst_mean.to_pandas()
sst_daily.index = pd.to_datetime(sst_daily.index)
sst_daily = sst_daily.sort_index()
sst_daily.name = "sst"

# --- monthly mean ---
sst_m = sst_daily.resample("MS").mean()  # month start
sst_m.name = "sst"

# --- anomaly: subtract month-of-year climatology ---
sst_anom_m = sst_m - sst_m.groupby(sst_m.index.month).transform("mean")

df_out = pd.DataFrame({"sst": sst_m, "sst_anom": sst_anom_m})
df_out.to_csv(out_csv, index=True)

print("Saved:", out_csv.resolve())
print("Range:", df_out.index.min(), "to", df_out.index.max(), "rows:", len(df_out))
print(df_out.head())

Found 13 files
First: oisst_socal_1984-01-01_1988-12-31.nc | Last: oisst_socal_test_2020.nc
Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_socal_bbox_monthly.csv
Range: 1984-01-01 00:00:00 to 2025-12-01 00:00:00 rows: 504
                  sst  sst_anom
time                           
1984-01-01  14.946989  0.861664
1984-02-01  14.871823  0.977874
1984-03-01  14.898540  1.227298
1984-04-01  13.516842  0.073137
1984-05-01  14.628971  0.585222


In [7]:
import pandas as pd
from pathlib import Path

sst_m_path = Path("../1_DATA/processed/oisst_NORCALv1_bbox_monthly.csv")
kelp_path  = Path("../1_DATA/processed/kelp_timeseries_norcalv1_bbox_quarterly.csv")
out_path   = Path("../1_DATA/processed/oisst_features_NORCAL_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

# load
sst_m = pd.read_csv(sst_m_path, parse_dates=["time"])
sst_m = sst_m.set_index("time").sort_index()

df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

# quarterly features from monthly
q = sst_m.resample("QS")  # quarter start buckets

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})

# lag (1 quarter)
feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

# align to kelp quarters (using q-start)
feat = feat.reindex(kelp_qstart)
feat.index = kelp_times  # set index to kelp timestamps for easy join later

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_features_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_q_mean_lag1
1984-02-15   14.905784        1.022278       1.227298                  NaN
1984-05-15   14.784969        0.499183       0.839190             1.022278
1984-08-15   19.527659        2.072631       2.553732             0.499183
1984-11-15   15.633710       -0.445956      -0.300495             2.072631
1985-02-15   13.353732       -0.529773      -0.279811            -0.445956
1985-05-15   13.924952       -0.360834      -0.229975            -0.529773
1985-08-15   17.502159        0.047131       0.271622            -0.360834
1985-11-15   15.583540       -0.496127      -0.152170             0.047131
1986-02-15   14.299319        0.415813       0.623964            -0.496127
1986-05-15   14.245347       -0.040439       0.557709             0.415813


In [1]:
import re, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import xarray as xr

# -----------------------
# PATHS / SETTINGS
# -----------------------
OUT_DIR = Path("../../1_DATA/raw")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TMP_DIR = OUT_DIR / "_tmp_oisst_daily"
TMP_DIR.mkdir(parents=True, exist_ok=True)

# NorCal bbox
lat_min, lat_max = 38.0, 40.0
lon_min, lon_max = -125.0, -123.0

START_DATE = "1984-01-01"
END_DATE   = "2025-12-31"

# NCEI static archive (daily files)
BASE = "https://www.ncei.noaa.gov/data/sea-surface-temperature-optimum-interpolation/v2.1/access/avhrr"

# speed knobs
DL_WORKERS   = 8
MAX_RETRIES  = 6

FNAME_RE = re.compile(r"oisst_norcal_(\d{4}-\d{2}-\d{2})_(\d{4}-\d{2}-\d{2})\.nc$")

def latest_subset_end(out_dir: Path):
    latest = None
    for p in out_dir.glob("oisst_norcal_*.nc"):
        m = FNAME_RE.match(p.name)
        if not m:
            continue
        if p.stat().st_size <= 0:
            continue
        end = pd.to_datetime(m.group(2))
        if latest is None or end > latest:
            latest = end
    return latest

def curl_fetch(url: str, out_path: Path):
    # fail fast + retry + IPv4 avoids common hangs
    cmd = [
        "curl", "-4",
        "-L", "--fail", "--compressed",
        "--connect-timeout", "20",
        "--max-time", "600",
        "--retry", str(MAX_RETRIES),
        "--retry-all-errors",
        "--retry-delay", "2",
        url,
        "-o", str(out_path),
    ]
    subprocess.run(cmd, check=True)

def download_day(d: pd.Timestamp) -> Path:
    ymd = d.strftime("%Y%m%d")
    ym  = d.strftime("%Y%m")
    url = f"{BASE}/{ym}/oisst-avhrr-v02r01.{ymd}.nc"
    out = TMP_DIR / f"oisst-avhrr-v02r01.{ymd}.nc"
    if out.exists() and out.stat().st_size > 0:
        return out
    part = out.with_suffix(out.suffix + ".part")
    try:
        curl_fetch(url, part)
        part.replace(out)
        return out
    finally:
        if part.exists():
            try: part.unlink()
            except: pass

def subset_month(files, out_nc: Path):
    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)

    # Handle lon convention (0..360 vs -180..180)
    if float(ds["lon"].max()) > 180:
        lmin = lon_min % 360
        lmax = lon_max % 360
        sub = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lmin, lmax))
    else:
        sub = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

    # Keep just what you need
    out = sub[["sst"]]

    # Write compressed
    out.to_netcdf(out_nc, encoding={"sst": {"zlib": True, "complevel": 4}})

    ds.close()
    sub.close()
    out.close()

# -----------------------
# RESUME LOGIC
# -----------------------
latest_end = latest_subset_end(OUT_DIR)
resume_start = (latest_end + pd.Timedelta(days=1)) if latest_end is not None else pd.to_datetime(START_DATE)
final_end = pd.to_datetime(END_DATE)

print("Latest subset end:", latest_end.date() if latest_end is not None else None)
print("Resuming from:", resume_start.date(), "to", final_end.date())

if resume_start > final_end:
    print("Nothing to do — already complete.")
    raise SystemExit

# month loop
month_starts = pd.date_range(resume_start.normalize().replace(day=1), final_end.normalize(), freq="MS")

for ms in month_starts:
    me = (ms + pd.offsets.MonthEnd(1)).to_pydatetime()
    month_start = max(pd.to_datetime(ms.date()), resume_start)
    month_end   = min(pd.to_datetime(me.date()), final_end)

    dates = pd.date_range(month_start, month_end, freq="D")
    out_nc = OUT_DIR / f"oisst_norcal_{month_start.strftime('%Y-%m-%d')}_{month_end.strftime('%Y-%m-%d')}.nc"

    if out_nc.exists() and out_nc.stat().st_size > 0:
        print("Skip subset (exists):", out_nc.name)
        continue

    print(f"\n=== {month_start.date()} to {month_end.date()} ({len(dates)} days) ===")

    # download daily files in parallel
    files = []
    with ThreadPoolExecutor(max_workers=DL_WORKERS) as ex:
        futs = [ex.submit(download_day, d) for d in dates]
        for i, f in enumerate(as_completed(futs), 1):
            files.append(f.result())
            if i % 5 == 0 or i == len(dates):
                print(f"Downloaded {i}/{len(dates)} days")

    files = sorted(files)  # ensure time order

    # subset + write monthly NorCal file
    print("Subsetting + writing:", out_nc.name)
    subset_month(files, out_nc)
    print("Saved:", out_nc, "bytes:", out_nc.stat().st_size)

    # cleanup tmp daily files for this month
    for fp in files:
        try: fp.unlink()
        except: pass

print("\n✅ Done. Monthly NorCal files written to:", OUT_DIR)

Latest subset end: 2005-01-29
Resuming from: 2005-01-30 to 2025-12-31

=== 2005-01-30 to 2005-01-31 (2 days) ===


  % To  % Total    % Received % Xferd  Average Spteed   Time    Time     Timea  Currentl
                        %   R e c e ived % Xferd  Ave       Dload rage  USppleoeadd      TTiomtea l      TSipmeen t        TLiemfet    CSuprereedn
t
     0           0         0           0         0           0D l o a d    0U p l o a d  0   -T-o:t-a-l: - -  S-p-e:n-t- : - -  L-e-f:t- - :Sp-e-e d 
100 1667k  100 1667k    0     0  1067k      0  0:00:01  0:00:01 --:--:-- 1071k
100 1667k  100 1667k    0     0   882k      0  0:00:01  0:00:01 --:--:--  885k


Downloaded 2/2 days
Subsetting + writing: oisst_norcal_2005-01-30_2005-01-31.nc
Saved: ../../1_DATA/raw/oisst_norcal_2005-01-30_2005-01-31.nc bytes: 38121

=== 2005-02-01 to 2005-02-28 (28 days) ===


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0  % Total    % Received % Xferd  Average Speed   Time   

Downloaded 5/28 days


...

Downloaded 20/31 days


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1527k  100 1527k    0     0  1305k      0  0:00:01  0:00:01 --:--:-- 1309k
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1528k  100 1528k    0     0  1126k      0  0:00:01  0:00:01 --:--:-- 1132k
100 1531k  100 1531k    0     0  1238k      0  0:00:01  0:00:01 --:--:-- 1244k
100 1531k  100 1531k    0     0  1319k      0  0:00:01  0:00:01 --:--:-- 1327k         0  0  0 :00:00:00:80 2  0 :00:00:001  0:00::01  0:00:01  721k07  187k
100 1532k  100 1532k    0     0  1256k      0  0:00:01  0:00:01 --:--:-- 1265k


Downloaded 25/31 days


100 1531k  100 1531k    0     0  1271k      0  0:00:01  0:00:01 --:--:-- 1278k
100 1530k  100 1530k    0     0  1197k      0  0:00:01  0:00:01 --:--:-- 1204k
100 1533k  100 1533k    0     0  1237k      0  0:00:01  0:00:01 --:--:-- 1244k
100 1530k  100 1530k    0     0  1155k      0  0:00:01  0:00:01 --:--:-- 1161k


Downloaded 30/31 days
Downloaded 31/31 days
Subsetting + writing: oisst_norcal_2025-12-01_2025-12-31.nc
Saved: ../../1_DATA/raw/oisst_norcal_2025-12-01_2025-12-31.nc bytes: 37937

✅ Done. Monthly NorCal files written to: ../../1_DATA/raw


In [2]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

RAW_DIR = Path("/Users/tonylin/Documents/kelp_project/1_DATA/raw")
OUT_CSV = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_norcalv1_bbox_monthly.csv")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

CLIM_START = "1991-01-01"
CLIM_END   = "2020-12-31"

files = sorted([p for p in RAW_DIR.glob("oisst_norcal_*.nc") if p.stat().st_size > 0])
print("Files:", len(files))
print("First/last:", files[0].name, files[-1].name)

def norm_ds(ds: xr.Dataset) -> xr.Dataset:
    # normalize dim/coord names
    if "latitude" in ds.dims:
        ds = ds.rename({"latitude": "lat"})
    if "longitude" in ds.dims:
        ds = ds.rename({"longitude": "lon"})
    if "latitude" in ds.coords and "lat" not in ds.coords:
        ds = ds.rename({"latitude": "lat"})
    if "longitude" in ds.coords and "lon" not in ds.coords:
        ds = ds.rename({"longitude": "lon"})
    # drop zlev if present
    if "zlev" in ds.dims:
        ds = ds.isel(zlev=0, drop=True)
    return ds

rows = []
for i, f in enumerate(files, 1):
    ds = xr.open_dataset(f)
    ds = norm_ds(ds)

    if "sst" not in ds:
        ds.close()
        continue

    # spatial mean over bbox grid
    s = ds["sst"].mean(dim=[d for d in ("lat", "lon") if d in ds["sst"].dims], skipna=True)
    ser = s.to_pandas()
    ser.name = "sst"
    rows.append(ser)

    ds.close()
    if i % 50 == 0 or i == len(files):
        print(f"Read {i}/{len(files)}")

# stitch -> one time series
sst_ts = pd.concat(rows).sort_index()
sst_ts = sst_ts[~sst_ts.index.duplicated(keep="first")]

# monthly means
sst_m = sst_ts.resample("MS").mean().to_frame()

# monthly climatology + anomaly
clim_slice = sst_m.loc[CLIM_START:CLIM_END, "sst"]
clim_by_month = clim_slice.groupby(clim_slice.index.month).mean()
sst_m["sst_anom"] = sst_m["sst"] - sst_m.index.month.map(clim_by_month)

# save
out = sst_m.reset_index().rename(columns={"index": "time"})
out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
print(out.head())

Files: 508
First/last: oisst_norcal_1984-01-01_1984-01-28.nc oisst_norcal_2025-12-01_2025-12-31.nc
Read 50/508
Read 100/508
Read 150/508
Read 200/508
Read 250/508
Read 300/508
Read 350/508
Read 400/508
Read 450/508
Read 500/508
Read 508/508
Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_norcalv1_bbox_monthly.csv
        time        sst  sst_anom
0 1984-01-01  12.414595  0.682843
1 1984-02-01  12.040409  0.380823
2 1984-03-01  12.074450  0.531710
3 1984-04-01  11.113460 -0.373553
4 1984-05-01  12.382489  0.577767


In [5]:
import pandas as pd
from pathlib import Path

sst_m_path = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_norcalv1_bbox_monthly.csv")
kelp_path  = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/kelp_timeseries_norcalv1_bbox.csv")
out_path   = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_features_norcal_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

sst_m = pd.read_csv(sst_m_path, parse_dates=["time"]).set_index("time").sort_index()

df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

q = sst_m.resample("QS")

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})

feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

feat = feat.reindex(kelp_qstart)
feat.index = kelp_times

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_features_norcal_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_q_mean_lag1
1984-02-15   12.176485        0.531792       0.682843                  NaN
1984-05-15   11.759291       -0.046608       0.577767             0.531792
1984-08-15   13.703316        0.221706       0.321739            -0.046608
1984-11-15   12.760182       -0.276030      -0.128920             0.221706
1985-02-15   10.754113       -0.890579       0.021274            -0.276030
1985-05-15   11.178831       -0.627068      -0.006830            -0.890579
1985-08-15   13.476942       -0.004669       0.148303            -0.627068
1985-11-15   12.074263       -0.961948      -0.541627            -0.004669
1986-02-15   12.390114        0.745422       1.148107            -0.961948
1986-05-15   11.843278        0.037379       0.719787             0.745422
